# Descrizione dell'esperimento
L'esperimento si divide in due fasi:
1. Addestramento e validazione in modalità **transductive** su grafi di AD e PD connessi, esattamente come in *test_topology_significance_connected.ipynb*. Vengono misurate le performance a livello di distribuzione
2. Validazione del modello $G_0$, addestrato quindi su grafo non rewired, su un dataset mai visto, prima lasciato invariato e poi alterato con rewiring
### Focus punto 2
##### Razionale
L'idea di fondo è quella di testare se il modello addestrato transduttivamente ha imparato a sfruttare delle feature topologiche particolari. Testandolo sui grafi rewired si testa la sua capacità di generalizzare in assenza di particolari feature che sono state alterate durante il processo di perturbazione.

Questo tipo di esperimento vede i rewiring come dei "generatori di dataset sintetici", interpretazione senza la quale si potrebbe pensare che stia introducendo un bias strutturale, in quanto i rewiring usano i label per perturbare in modo controllato i grafi. 
##### Experimental setting
Vista la randomicità del processo di rewiring, bisogna campionare $N$ grafi perturbati per ogni distribuzione generata. Collezionando le performance per ognuno di questi grafi, si farà una comparazione statistica rispetto ai risultati del punto 1.
##### Criteri di successo
1. La distribuzione è identica: molto improbabile, ma dimostrerebbe definitivamente che il modello ha imparato a sfruttare le feature e che l'ablazione su dataset ha un effetto predicibile
2. La distribuzione è "proporzionata": si intende che ordinando le performance di GCN_plain, R1, R2, R3 e MLP si abbia lo stesso ordine del punto 1. Anche questo sarebbe un ottimo risultato, nonostante le performance scendano.

# Parte 1 - Transductive training

In [2]:
%load_ext autoreload
%autoreload 2

from data_pipeline import *
import torch_geometric.transforms as T

from training_tools.splitter import StratifiedSplitterWithTestHoldout, HoldoutMode
from training_tools.task import TaskBinaryNodeClassification
from training_tools.trainer import Trainer
from training_tools.validation import CrossValidator
from training_tools.evaluation import Evaluator

from models import GCN, GCN_Sage, MLP

from tqdm import tqdm


pre_transform = T.Compose([
    MarkCommonNodes(edge_types=('AD','PD')),
    ReindexConsecutive(),
    MergeRelationsToHomogeneous(merge=['AD','PD'], edge_attr_reduce='sum'),
    FilterSmallConnectedComponents(topk=1)
])

ds = NeuroDegAnc2VecDataset(root='data', pre_transform=pre_transform, force_reload=True)
data = ds[0]

Processing...
Done!


In [3]:
def compare_models_over_repeats(
    data,                   # Data
    model_factories,           # [factory_A, factory_B, ...]
    splitter_ctor,             # lambda seed, mode: StratifiedNestedSplitter(...)
    task, trainer, mode="transductive",
    n = 10
):
    seeds=list(range(n, 2*n))
    # metriche per modello: dict[name] -> list of per-run test F1
    results = {i: [] for i in range(len(model_factories))}
    artifacts_list = []

    with tqdm(total=len(seeds), desc="Training Progress", unit="iteration") as pbar:
        for seed in seeds:
            splitter = splitter_ctor(seed, mode)
            parts = splitter.split(data, holdout_mode=HoldoutMode.RANDOM, aux_construction_property='is_common', aux_in_test=False)

            for idx, factory in enumerate(model_factories):
                cv = CrossValidator(trainer=trainer, model_factory=factory, mode=mode, select_by="f1")
                artifacts, summary = cv.run(task, data, parts)

                evaluator = Evaluator(model_factory=factory, mode=mode)
                test_metrics = evaluator.evaluate(task, data, parts, artifacts["best_state"])
                results[idx].append(test_metrics["f1"])  # metrica primaria
                artifacts_list.append(artifacts)
            pbar.update(1)

    return results, artifacts_list

In [4]:
from graph_data_manipulation import rewire_degree_preserving_complete, rewire_degree_preserving_stratified_connected, rewire_non_degree_preserving_stratified_complete

mode = 'transductive'
seed = 42
gcn_factory = lambda : GCN(in_channels=data.x.size(1), hidden_channels=64, out_channels=1, dropout=0)
mlp_factory = lambda : MLP(in_channels=data.x.size(1), hidden_channels=64, out_channels=1, dropout=0)
splitter_ctor = lambda seed, mode: StratifiedSplitterWithTestHoldout(test_size=0.1, n_splits=5, seed=seed, mode=mode)


task    = TaskBinaryNodeClassification(threshold=0.5, pos_weight=None)
trainer = Trainer(lr=1e-3, weight_decay=5e-4, max_epochs=50, patience=10)

n = 10  # Numero di training
m = 20  # Numero di rewiring

rewiring_algs = {
    "R1": rewire_degree_preserving_complete,
    "R2": rewire_degree_preserving_stratified_connected,
    "R3": rewire_non_degree_preserving_stratified_complete
}

for name, alg in rewiring_algs.items():
    print(f"{name} | {alg.__name__}")

training_metrics = dict() # key: `model's name`; value: List[artifact]

# --- Training MLP
test_metrics_mlp, artifacts_mlp = compare_models_over_repeats(
    data, 
    [mlp_factory], 
    splitter_ctor, task, trainer, mode='transductive', n=n)

training_metrics['MLP'] = test_metrics_mlp[0]

# --- Training GCN
test_metrics_gcn = []
gcn_artifacts = list()
for x in tqdm(range(n*m), desc="GCN training", unit="trainings"): 
    splitter = StratifiedSplitterWithTestHoldout(test_size=0.1, n_splits=5, seed=seed + x, mode=mode)
    parts = splitter.split(data, holdout_mode=HoldoutMode.RANDOM, aux_construction_property='is_common', aux_in_test=False)

    cv = CrossValidator(trainer=trainer, model_factory=gcn_factory, mode=mode, select_by="f1")
    artifacts, summary = cv.run(task, data, parts)

    gcn_artifacts.append(artifacts)

    evaluator = Evaluator(model_factory=gcn_factory, mode=mode)
    test_metrics_gcn.append(evaluator.evaluate(task, data, parts, artifacts["best_state"])['f1'])

training_metrics['GCN'] = test_metrics_gcn

# --- Training GCN on rewiring
progress_bar = tqdm(total=n*m*len(rewiring_algs), unit="trainings")
test_metrics_rew = dict()
for alg_name in rewiring_algs.keys():
    test_metrics_rew[alg_name] = list()

rew_count = 0
for alg_name, rew_alg in rewiring_algs.items():
    rew_count += 1
    rew_artifacts_list = list()
    for i in range(m):
        progress_bar.set_description(f"[{rew_count}] [{alg_name}] Training on {i+1}th rewiring")
        g_rewired = rew_alg(
            data,
            seed=seed + i
        )

        #for x in tqdm.tqdm(range(n), desc="Training GCN on original and rewired graphs", unit="trainings"):
        for j in range(n):
            splitter = StratifiedSplitterWithTestHoldout(test_size=0.1, n_splits=5, seed=seed + j, mode=mode)
            rewired_parts = splitter.split(g_rewired, holdout_mode=HoldoutMode.RANDOM, aux_construction_property='is_common', aux_in_test=False)

            cv_rew = CrossValidator(trainer=trainer, model_factory=gcn_factory, mode=mode, select_by="f1")
            artifacts_rew, summary_rew = cv_rew.run(task, g_rewired, rewired_parts)

            rew_artifacts_list.append(artifacts_rew)

            evaluator = Evaluator(model_factory=gcn_factory, mode=mode)
            test_metrics_rew[alg_name].append(evaluator.evaluate(task, g_rewired, rewired_parts, artifacts_rew["best_state"])['f1'])

            progress_bar.update(1)
        
        progress_bar.update(1)

training_metrics = {**training_metrics, **test_metrics_rew}

import json
with open('scores.json', 'w', encoding='utf-8') as f:
    json.dump(training_metrics, f, indent=4)

with open('best_state.json', 'w', encoding='utf-8') as f:
    json.dump(artifacts['best_state'], f, indent=4)

R1 | rewire_degree_preserving_complete
R2 | rewire_degree_preserving_stratified_connected
R3 | rewire_non_degree_preserving_stratified_complete


GCN training: 100%|██████████| 200/200 [03:59<00:00,  1.20s/trainings]
[3] [R3] Training on 20th rewiring: : 659trainings [10:40,  1.20s/trainings]                

TypeError: Object of type Tensor is not JSON serializable

In [5]:
from pathlib import Path

project_root = Path('.')
data_root = project_root / "data"

class_labels = (('T1', 'T2'))

c0_nodes_csv = project_root / "networks" / "MS_nodes.csv"
c1_nodes_csv = project_root / "networks" / "SLE_nodes.csv"
c0_edges_csv = project_root / "networks" / "MS_edges.csv"
c1_edges_csv = project_root / "networks" / "SLE_edges.csv"

pre_transform = T.Compose([
    MarkCommonNodes(edge_types=class_labels),
    ReindexConsecutive(),
    MergeRelationsToHomogeneous(merge=class_labels, edge_attr_reduce='sum'),
    # FilterSmallConnectedComponents(topk=1),
    FilterLargestComponentPerClass(class_labels=(0, 1))
])

test_ds = NeuroDegAnc2VecDataset(
    root=str(data_root), # Directory where processed data (hetero.pt) will be stored
    c0_nodes_csv=str(c0_nodes_csv),
    c1_nodes_csv=str(c1_nodes_csv),
    c0_edges_csv=str(c0_edges_csv),
    c1_edges_csv=str(c1_edges_csv),
    force_reload=True, # Set to True to re-process if needed, False to load from cache
    class_labels=class_labels,
    transform=None,
    pre_transform=pre_transform
)[0]

Processing...
Done!


In [ ]:
(test_ds.y == 0).count_nonzero()

tensor(17)

In [7]:
import torch
import numpy as np
from tqdm import tqdm
import json
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report


def evaluate(task, data, mask, state_dict):
    model = gcn_factory()
    model.load_state_dict(state_dict)
    model.eval()

    parts = {
        "supervised_mask":  torch.as_tensor(mask, dtype=torch.bool, device=data.y.device),
        "test_mask":        torch.as_tensor(mask, dtype=torch.bool, device=data.y.device),
    }

    y_true, y_pred = task.predict(model, data, parts, phase="test", fold_k=None, mode='transductive')

    # For flipped prediction
    #y_pred = 1-y_pred

    yt = y_true.view(-1).cpu().numpy()
    yp = y_pred.view(-1).cpu().numpy()

    acc = accuracy_score(yt, yp)
    prec, rec, f1, _ = precision_recall_fscore_support(yt, yp, average="binary", zero_division=0)

    return {
            "accuracy": float(acc), 
            "precision": float(prec), 
            "recall": float(rec), 
            "f1": float(f1)
    }

# Import delle strategie di rewiring dal modulo dedicato
from graph_data_manipulation.rewiring import (
    rewire_degree_preserving_complete,                # R1
    rewire_degree_preserving_stratified_connected,    # R2
    rewire_non_degree_preserving_stratified_complete  # R3
)
from training_tools.evaluation.Evaluator import Evaluator

# --- SETUP DELL'ESPERIMENTO ---
# Assunzioni:
# - model_factory: la funzione che crea l'architettura GCN
# - best_state_dict: i pesi del miglior modello addestrato sul grafo reale
# - T: l'oggetto PyG Data del dataset esterno
# - task: l'istanza di AbstractTask usata nel training

NUM_ROUNDS = 20
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Inizializziamo l'Evaluator in modalità transductive 
# (perché passiamo l'intero grafo T/T1/2/3 al forward del modello)
evaluator = Evaluator(model_factory=gcn_factory, mode='transductive')

# Poiché T è un dataset esterno, vogliamo valutare tutti i suoi nodi
num_nodes_T = test_ds.y.size(0)
full_mask = torch.ones(num_nodes_T, dtype=torch.bool, device=test_ds.y.device) & ~(test_ds.y == 2)

# Dizionario per collezionare le metriche
test_metrics_results = {}

# 1. Inferenza sul Test Set Originale (T)
test_metrics_results["T"] = evaluate(task, test_ds, full_mask, artifacts['best_state'])

# 2. Definizione delle strategie di Rewiring
rewiring_strategies = {
    "T1": rewire_degree_preserving_complete,
    "T2": rewire_degree_preserving_stratified_connected,
    "T3": rewire_non_degree_preserving_stratified_complete
}

# 3. Ciclo di valutazione sui Null Models (T1, T2, T3)

full_test_mask = (test_ds.y == 0) | (test_ds.y == 1)
for label, rewire_fn in rewiring_strategies.items():
    print(f"Esecuzione {NUM_ROUNDS} round per la strategia {label}...")
    rounds_data = []
    
    for r in tqdm(range(NUM_ROUNDS)):
        # Generiamo il grafo sintetico con un seed diverso per ogni round
        # Usiamo data.clone() per sicurezza all'interno delle funzioni di rewiring
        T_synth = rewire_fn(test_ds, seed=r)
        
        # Calcoliamo le metriche usando l'evaluator
        m = evaluate(
            task=task,
            data=T_synth,
            mask=full_test_mask,
            state_dict=artifacts['best_state']
        )
        rounds_data.append(m)
    
    # Aggregazione statistica dei risultati
    metric_keys = ["accuracy", "precision", "recall", "f1"]
    summary = {}
    for k in metric_keys:
        values = [res[k] for res in rounds_data]
        summary[f"{k}_mean"] = float(np.mean(values))
        summary[f"{k}_std"] = float(np.std(values))
    
    test_metrics_results[label] = summary

# Output finale dei risultati
print("\n--- RISULTATI FINALI ---")
print(json.dumps(test_metrics_results, indent=4))


Esecuzione 20 round per la strategia T1...


100%|██████████| 20/20 [00:00<00:00, 94.61it/s]


Esecuzione 20 round per la strategia T2...


100%|██████████| 20/20 [00:00<00:00, 111.57it/s]


Esecuzione 20 round per la strategia T3...


100%|██████████| 20/20 [00:00<00:00, 155.04it/s]


--- RISULTATI FINALI ---
{
    "T": {
        "accuracy": 0.5897435897435898,
        "precision": 0.625,
        "recall": 0.6818181818181818,
        "f1": 0.6521739130434783
    },
    "T1": {
        "accuracy_mean": 0.5115384615384615,
        "accuracy_std": 0.06806955873055531,
        "precision_mean": 0.5801887894919426,
        "precision_std": 0.07277425771553982,
        "recall_mean": 0.4909090909090909,
        "recall_std": 0.11463200193562263,
        "f1_mean": 0.5260755773787469,
        "f1_std": 0.08495774133296188
    },
    "T2": {
        "accuracy_mean": 0.5205128205128207,
        "accuracy_std": 0.0973346302364221,
        "precision_mean": 0.606506050708837,
        "precision_std": 0.11304118624845151,
        "recall_mean": 0.43636363636363634,
        "recall_std": 0.11535070491317746,
        "f1_mean": 0.5024218197941964,
        "f1_std": 0.10796992181046426
    },
    "T3": {
        "accuracy_mean": 0.5474358974358974,
        "accuracy_std": 0.12463